# PulseTMS — UPV Forecast Bronze Upload

**Purpose:** Upload a quarterly UPV forecast CSV to the Bronze table in BigQuery.  
**Target table:** `prj-dbi-prd-1.ds_dbi_digitalmedia_automation.sdi_pulseTms_bronze_upvForecast_weekly`  
**Upsert logic:** New weeks are inserted; existing weeks are overwritten with the latest forecast values; weeks not in the upload are untouched.  

**Expected CSV columns:**
| Column | Description |
|--------|-------------|
| `Week Ending` | Saturday date (or corrected quarter-end date for boundary stubs) |
| `2026 Forecast` | UPV forecast (maps to `upvTotalAdobe`) |
| `2026 Web +App Forecast` | UPV Web + App forecast |

> **Before uploading:** Ensure boundary stub weeks use the correct quarter-end date (e.g. `9/30/2026`), not the following Saturday.

## 1. Imports & Configuration

In [ ]:
import io
import re
import warnings
from datetime import date

import ipywidgets as widgets
import pandas as pd
from google.cloud import bigquery
from IPython.display import display, HTML

warnings.filterwarnings('ignore')

# ── Configuration ────────────────────────────────────────────────────────────
PROJECT_ID  = 'prj-dbi-prd-1'
DATASET_ID  = 'ds_dbi_digitalmedia_automation'
TABLE_ID    = 'sdi_pulseTms_bronze_upvForecast_weekly'
FULL_TABLE  = f'{PROJECT_ID}.{DATASET_ID}.{TABLE_ID}'

# Column name mapping: CSV → Bronze
COL_MAP = {
    'Week Ending'            : 'week_sun_sat',
    '2026 Forecast'          : 'upv_forecast',
    '2026 Web +App Forecast' : 'upv_webapp_forecast',
}

# BigQuery client — uses Vertex AI Workbench ADC automatically
client = bigquery.Client(project=PROJECT_ID)

print(f'✅ Connected to project: {PROJECT_ID}')
print(f'   Target table: {FULL_TABLE}')

## 2. Upload Widget

In [ ]:
uploader = widgets.FileUpload(
    accept='.csv',
    multiple=False,
    description='Upload CSV',
    layout=widgets.Layout(width='300px')
)

display(
    HTML('<h4 style="font-family:sans-serif; margin-bottom:8px">Select quarterly UPV forecast CSV</h4>'),
    uploader
)

## 3. Parse & Validate CSV
Run this cell after uploading the file above.

In [ ]:
# ── Guard: ensure a file was uploaded ────────────────────────────────────────
if not uploader.value:
    raise ValueError('No file uploaded. Please use the widget above first.')

# ── Read uploaded bytes into a DataFrame ─────────────────────────────────────
uploaded_file = list(uploader.value.values())[0]
filename      = list(uploader.value.keys())[0]
raw_bytes     = uploaded_file['content']

df_raw = pd.read_csv(io.BytesIO(raw_bytes))
print(f'📄 File loaded: {filename}')
print(f'   Raw shape: {df_raw.shape[0]} rows × {df_raw.shape[1]} columns')
print(f'   Columns found: {list(df_raw.columns)}')

# ── Validate expected columns exist ──────────────────────────────────────────
missing_cols = [c for c in COL_MAP.keys() if c not in df_raw.columns]
if missing_cols:
    raise ValueError(
        f'Missing expected columns: {missing_cols}\n'
        f'Columns in file: {list(df_raw.columns)}'
    )

# ── Select and rename columns ─────────────────────────────────────────────────
df = df_raw[list(COL_MAP.keys())].copy()
df.rename(columns=COL_MAP, inplace=True)

# ── Parse week_sun_sat as DATE ────────────────────────────────────────────────
df['week_sun_sat'] = pd.to_datetime(df['week_sun_sat'], infer_datetime_format=True).dt.date

# ── Parse metric columns — strip commas/spaces, cast to float ─────────────────
for col in ['upv_forecast', 'upv_webapp_forecast']:
    df[col] = (
        df[col]
        .astype(str)
        .str.replace(',', '', regex=False)
        .str.strip()
        .pipe(pd.to_numeric, errors='coerce')
    )

# ── Add metadata columns ──────────────────────────────────────────────────────
df['file_load_date'] = date.today()
df['source_filename'] = filename

# ── Drop rows where week_sun_sat is null ──────────────────────────────────────
null_dates = df['week_sun_sat'].isna().sum()
if null_dates > 0:
    print(f'⚠️  Dropping {null_dates} rows with null week_sun_sat')
    df = df.dropna(subset=['week_sun_sat'])

# ── Preview ───────────────────────────────────────────────────────────────────
print(f'\n✅ Parsed successfully — {len(df)} rows ready to upsert')
print(f'   Date range: {df["week_sun_sat"].min()} → {df["week_sun_sat"].max()}')
print()
display(df.head(10))

## 4. Validate Dates
Checks that each date is either a Saturday or a known quarter-end date. Flags anything unexpected so you can fix the CSV before writing to BQ.

In [ ]:
import calendar

# Quarter-end dates are Mar 31, Jun 30, Sep 30, Dec 31
QUARTER_END_MONTHS_DAYS = {(3, 31), (6, 30), (9, 30), (12, 31)}

issues = []
for _, row in df.iterrows():
    d = row['week_sun_sat']
    is_saturday     = d.weekday() == 5  # Monday=0, Saturday=5
    is_quarter_end  = (d.month, d.day) in QUARTER_END_MONTHS_DAYS
    if not is_saturday and not is_quarter_end:
        issues.append({
            'week_sun_sat' : d,
            'day_of_week'  : calendar.day_name[d.weekday()],
            'issue'        : 'Not a Saturday or quarter-end date'
        })

if issues:
    print(f'⚠️  {len(issues)} date(s) failed validation:')
    display(pd.DataFrame(issues))
    print('\nPlease fix these dates in the CSV before proceeding.')
    print('Boundary stub weeks should use the quarter-end date (e.g. 9/30/2026), not the following Saturday.')
else:
    print(f'✅ All {len(df)} dates are valid (Saturday or quarter-end).')

## 5. Upsert to BigQuery

**Logic:**
1. Write incoming rows to a temporary staging table
2. `MERGE` staging → Bronze on `week_sun_sat`
   - **MATCH:** overwrite forecast values + update `file_load_date` and `source_filename`
   - **NO MATCH:** insert new row
   - Weeks not in the upload are untouched

> Only run this cell once you're happy with the preview and validation above.

In [ ]:
# ── Abort if date validation found issues ─────────────────────────────────────
if issues:
    raise ValueError(
        f'{len(issues)} date validation issue(s) found. '
        'Fix the CSV and re-run from Cell 3 before writing to BigQuery.'
    )

STAGING_TABLE = f'{FULL_TABLE}_staging_temp'

# ── Step 1: Write to staging table (full replace) ─────────────────────────────
print('📤 Writing to staging table...')

job_config = bigquery.LoadJobConfig(
    write_disposition = bigquery.WriteDisposition.WRITE_TRUNCATE,
    schema = [
        bigquery.SchemaField('week_sun_sat',        'DATE'),
        bigquery.SchemaField('upv_forecast',        'FLOAT64'),
        bigquery.SchemaField('upv_webapp_forecast', 'FLOAT64'),
        bigquery.SchemaField('file_load_date',      'DATE'),
        bigquery.SchemaField('source_filename',     'STRING'),
    ]
)

load_job = client.load_table_from_dataframe(df, STAGING_TABLE, job_config=job_config)
load_job.result()  # Wait for completion
print(f'   ✅ Staging table written: {STAGING_TABLE} ({len(df)} rows)')

# ── Step 2: MERGE staging → Bronze ────────────────────────────────────────────
print('\n🔀 Running MERGE into Bronze table...')

merge_sql = f"""
MERGE `{FULL_TABLE}` AS target
USING `{STAGING_TABLE}` AS source
  ON target.week_sun_sat = source.week_sun_sat

-- Existing week: overwrite forecast values, update metadata
WHEN MATCHED THEN UPDATE SET
  target.upv_forecast        = source.upv_forecast,
  target.upv_webapp_forecast = source.upv_webapp_forecast,
  target.file_load_date      = source.file_load_date,
  target.source_filename     = source.source_filename

-- New week: insert full row
WHEN NOT MATCHED BY TARGET THEN INSERT (
  week_sun_sat,
  upv_forecast,
  upv_webapp_forecast,
  file_load_date,
  source_filename
) VALUES (
  source.week_sun_sat,
  source.upv_forecast,
  source.upv_webapp_forecast,
  source.file_load_date,
  source.source_filename
)
"""

merge_job = client.query(merge_sql)
merge_job.result()  # Wait for completion

print(f'   ✅ MERGE complete')
print(f'   Rows affected: {merge_job.num_dml_affected_rows}')

# ── Step 3: Drop staging table ────────────────────────────────────────────────
client.delete_table(STAGING_TABLE, not_found_ok=True)
print(f'   🗑️  Staging table dropped')

## 6. Verify — Preview Bronze Table

In [ ]:
verify_sql = f"""
SELECT
  week_sun_sat,
  upv_forecast,
  upv_webapp_forecast,
  file_load_date,
  source_filename
FROM `{FULL_TABLE}`
ORDER BY week_sun_sat DESC
LIMIT 20
"""

df_verify = client.query(verify_sql).to_dataframe()

print(f'📊 Bronze table preview (most recent 20 rows):')
print(f'   Total rows in table: ', end='')

count_sql  = f'SELECT COUNT(*) AS n FROM `{FULL_TABLE}`'
total_rows = client.query(count_sql).to_dataframe()['n'].iloc[0]
print(total_rows)
print(f'   Date range in table: ', end='')

range_sql = f'SELECT MIN(week_sun_sat) AS min_dt, MAX(week_sun_sat) AS max_dt FROM `{FULL_TABLE}`'
range_df  = client.query(range_sql).to_dataframe()
print(f"{range_df['min_dt'].iloc[0]} → {range_df['max_dt'].iloc[0]}")
print()
display(df_verify)